In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!nvidia-smi


Fri Feb 20 03:41:47 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   34C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
%cd /content/drive/MyDrive/GB-YOLOv7/yolov7

## Train and test with GB_YOLOv7




In [ ]:
import os
os.environ["WANDB_MODE"] = "disabled"

In [ ]:
!python train.py  --device 0 --batch-size 16 --epoch  100 --img 640 640 --data data/data.yaml --hyp data/hyp.scratch.custom.yaml --cfg cfg/training/GB-architecture.yaml --weights yolov7.pt --name gb_train_17-02-2026

2026-02-17 01:43:41.331216: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1771292621.352067    1670 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1771292621.358957    1670 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1771292621.377388    1670 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1771292621.377416    1670 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1771292621.377421    1670 computation_placer.cc:177] computation placer alr

**Testing and validation**

In [ ]:
!python test.py --data data/data.yaml --img 640 --batch 32 --conf 0.001 --iou 0.65 --weights runs/train/gb_train_17-02-2026/weights/best.pt --name gb_output


Namespace(weights=['runs/train/gb_train_17-02-2026/weights/best.pt'], data='data/data.yaml', batch_size=32, img_size=640, conf_thres=0.001, iou_thres=0.65, task='val', device='', single_cls=False, augment=False, verbose=False, save_txt=False, save_hybrid=False, save_conf=False, save_json=False, project='runs/test', name='gb_output', exist_ok=False, no_trace=False, v5_metric=False)
YOLOR 🚀 2026-2-17 torch 2.9.0+cu128 CUDA:0 (Tesla T4, 14912.6875MB)

Fusing layers... 
IDetect.fuse
Model Summary: 336 layers, 31045264 parameters, 0 gradients
 Convert model to Traced-model... 
 traced_script_module saved! 
 model is traced! 

/usr/local/lib/python3.12/dist-packages/torch/functional.py:505: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4317.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]
val: Scanning '/content/drive/MyDrive/GB-YOLOv7/yolov

In [ ]:
!find /content/drive/MyDrive/GB-YOLOv7 -name "*.cache" -print

In [ ]:
import os, time, glob, sys
import cv2
import numpy as np
import torch

# -------- SETTINGS --------
IMG_DIR = "/content/drive/MyDrive/GB-YOLOv7/yolov7/dataset-yolo_final/temp/images"
WEIGHTS = "/content/drive/MyDrive/GB-YOLOv7/yolov7/yolov7.pt"  # or your runs/train/.../best.pt
YOLOV7_REPO = "/content/drive/MyDrive/GB-YOLOv7/yolov7"        # yolov7 repo root (must contain models/)
IMG_SIZE = 640
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
N_WARMUP = 30
N_RUN = 200
INCLUDE_PREPROCESS = True
# --------------------------

# Make YOLOv7 code importable
if YOLOV7_REPO not in sys.path:
    sys.path.append(YOLOV7_REPO)

from models.experimental import attempt_load  # YOLOv7 loader

# Speed/stability
torch.backends.cudnn.benchmark = True

def letterbox(img, size=640):
    """Simple square letterbox (keeps aspect ratio, pads to size x size)."""
    h, w = img.shape[:2]
    scale = min(size / h, size / w)
    nh, nw = int(round(h * scale)), int(round(w * scale))
    resized = cv2.resize(img, (nw, nh), interpolation=cv2.INTER_LINEAR)
    canvas = np.full((size, size, 3), 114, dtype=np.uint8)
    canvas[:nh, :nw] = resized
    return canvas

def preprocess(img):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    gray = cv2.medianBlur(gray, 5)
    clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))
    gray = clahe.apply(gray)
    return cv2.cvtColor(gray, cv2.COLOR_GRAY2BGR)

# Load images
files = sorted(glob.glob(os.path.join(IMG_DIR, "*")))
files = files[:min(N_RUN, len(files))]
if len(files) == 0:
    raise FileNotFoundError(f"No images found in: {IMG_DIR}")

# ---- Load model (PyTorch 2.6+ safe way for YOLOv7) ----
model = attempt_load(WEIGHTS, map_location=DEVICE)  # handles yolov7 checkpoints
model.eval().to(DEVICE)

# Warm-up
dummy = torch.zeros((1, 3, IMG_SIZE, IMG_SIZE), device=DEVICE)
with torch.no_grad():
    for _ in range(N_WARMUP):
        out = model(dummy)
        # touch output to ensure full compute path
        _ = out[0] if isinstance(out, (list, tuple)) else out
if DEVICE.startswith("cuda"):
    torch.cuda.synchronize()

times = []
with torch.no_grad():
    for fp in files:
        img = cv2.imread(fp)
        if img is None:
            continue

        t0 = time.perf_counter()

        if INCLUDE_PREPROCESS:
            img = preprocess(img)

        img = letterbox(img, IMG_SIZE)

        # BGR->RGB, HWC->CHW
        img = img[:, :, ::-1].transpose(2, 0, 1)
        img = np.ascontiguousarray(img)

        img = torch.from_numpy(img).to(DEVICE).float() / 255.0
        img = img.unsqueeze(0)  # (1,3,H,W)

        out = model(img)  # forward only
        _ = out[0] if isinstance(out, (list, tuple)) else out

        if DEVICE.startswith("cuda"):
            torch.cuda.synchronize()
        t1 = time.perf_counter()

        times.append((t1 - t0) * 1000)

times = np.array(times, dtype=np.float64)
print("Images measured:", len(times))
print("Mean latency (ms):", times.mean())
print("FPS:", 1000.0 / times.mean())
print("p50 (ms):", np.percentile(times, 50))
print("p90 (ms):", np.percentile(times, 90))
print("p99 (ms):", np.percentile(times, 99))


Download error: [Errno 2] No such file or directory: '/content/drive/mydrive/gb-yolov7/yolov7/yolov7.pt.67b34dcb2d7947fda58ea6b0b4ffc854.partial'
ERROR: Download failure: /content/drive/mydrive/gb-yolov7/yolov7/yolov7.pt missing, try downloading from https://github.com/WongKinYiu/yolov7/releases/

Fusing layers... 
RepConv.fuse_repvgg_block
RepConv.fuse_repvgg_block
RepConv.fuse_repvgg_block


/usr/local/lib/python3.12/dist-packages/torch/nn/modules/module.py:969: UserWarning: The .grad attribute of a Tensor that is not a leaf Tensor is being accessed. Its .grad attribute won't be populated during autograd.backward(). If you indeed want the .grad field to be populated for a non-leaf Tensor, use .retain_grad() on the non-leaf Tensor. If you access the non-leaf Tensor by mistake, make sure you access the leaf Tensor instead. See github.com/pytorch/pytorch/pull/30531 for more information. (Triggered internally at /pytorch/build/aten/src/ATen/core/TensorBody.h:489.)
  param_grad = param.grad
/usr/local/lib/python3.12/dist-packages/torch/functional.py:505: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4317.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


Images measured: 55
Mean latency (ms): 63.36146665454924
FPS: 15.782462951056102
p50 (ms): 62.92876900010924
p90 (ms): 68.41216019997773
p99 (ms): 72.29357881999022


In [ ]:
import os, time, glob, sys
import cv2
import numpy as np
import torch

# -------- SETTINGS --------
IMG_DIR = "/content/drive/MyDrive/GB-YOLOv7/yolov7/dataset-yolo_final/temp/images"
WEIGHTS = "/content/drive/MyDrive/GB-YOLOv7/yolov7/yolov7.pt"  # or your runs/train/.../best.pt
YOLOV7_REPO = "/content/drive/MyDrive/GB-YOLOv7/yolov7"        # yolov7 repo root (must contain models/)
IMG_SIZE = 640
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
N_WARMUP = 30
N_RUN = 200
INCLUDE_PREPROCESS = False
# --------------------------

# Make YOLOv7 code importable
if YOLOV7_REPO not in sys.path:
    sys.path.append(YOLOV7_REPO)

from models.experimental import attempt_load  # YOLOv7 loader

# Speed/stability
torch.backends.cudnn.benchmark = True

def letterbox(img, size=640):
    """Simple square letterbox (keeps aspect ratio, pads to size x size)."""
    h, w = img.shape[:2]
    scale = min(size / h, size / w)
    nh, nw = int(round(h * scale)), int(round(w * scale))
    resized = cv2.resize(img, (nw, nh), interpolation=cv2.INTER_LINEAR)
    canvas = np.full((size, size, 3), 114, dtype=np.uint8)
    canvas[:nh, :nw] = resized
    return canvas

def preprocess(img):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    gray = cv2.medianBlur(gray, 5)
    clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))
    gray = clahe.apply(gray)
    return cv2.cvtColor(gray, cv2.COLOR_GRAY2BGR)

# Load images
files = sorted(glob.glob(os.path.join(IMG_DIR, "*")))
files = files[:min(N_RUN, len(files))]
if len(files) == 0:
    raise FileNotFoundError(f"No images found in: {IMG_DIR}")

# ---- Load model (PyTorch 2.6+ safe way for YOLOv7) ----
model = attempt_load(WEIGHTS, map_location=DEVICE)  # handles yolov7 checkpoints
model.eval().to(DEVICE)

# Warm-up
dummy = torch.zeros((1, 3, IMG_SIZE, IMG_SIZE), device=DEVICE)
with torch.no_grad():
    for _ in range(N_WARMUP):
        out = model(dummy)
        # touch output to ensure full compute path
        _ = out[0] if isinstance(out, (list, tuple)) else out
if DEVICE.startswith("cuda"):
    torch.cuda.synchronize()

times = []
with torch.no_grad():
    for fp in files:
        img = cv2.imread(fp)
        if img is None:
            continue

        t0 = time.perf_counter()

        if INCLUDE_PREPROCESS:
            img = preprocess(img)

        img = letterbox(img, IMG_SIZE)

        # BGR->RGB, HWC->CHW
        img = img[:, :, ::-1].transpose(2, 0, 1)
        img = np.ascontiguousarray(img)

        img = torch.from_numpy(img).to(DEVICE).float() / 255.0
        img = img.unsqueeze(0)  # (1,3,H,W)

        out = model(img)  # forward only
        _ = out[0] if isinstance(out, (list, tuple)) else out

        if DEVICE.startswith("cuda"):
            torch.cuda.synchronize()
        t1 = time.perf_counter()

        times.append((t1 - t0) * 1000)

times = np.array(times, dtype=np.float64)
print("Images measured:", len(times))
print("Mean latency (ms):", times.mean())
print("FPS:", 1000.0 / times.mean())
print("p50 (ms):", np.percentile(times, 50))
print("p90 (ms):", np.percentile(times, 90))
print("p99 (ms):", np.percentile(times, 99))


Download error: [Errno 2] No such file or directory: '/content/drive/mydrive/gb-yolov7/yolov7/yolov7.pt.df7d673f748948e0838de864e24c4e83.partial'
ERROR: Download failure: /content/drive/mydrive/gb-yolov7/yolov7/yolov7.pt missing, try downloading from https://github.com/WongKinYiu/yolov7/releases/

Fusing layers... 
RepConv.fuse_repvgg_block
RepConv.fuse_repvgg_block
RepConv.fuse_repvgg_block


/usr/local/lib/python3.12/dist-packages/torch/nn/modules/module.py:969: UserWarning: The .grad attribute of a Tensor that is not a leaf Tensor is being accessed. Its .grad attribute won't be populated during autograd.backward(). If you indeed want the .grad field to be populated for a non-leaf Tensor, use .retain_grad() on the non-leaf Tensor. If you access the non-leaf Tensor by mistake, make sure you access the leaf Tensor instead. See github.com/pytorch/pytorch/pull/30531 for more information. (Triggered internally at /pytorch/build/aten/src/ATen/core/TensorBody.h:489.)
  param_grad = param.grad


Images measured: 55
Mean latency (ms): 29.882633399998586
FPS: 33.464252852630025
p50 (ms): 30.02765899998394
p90 (ms): 30.748493599958238
p99 (ms): 30.974651899971377


In [ ]:
import os, time, glob, sys
import cv2
import numpy as np
import torch

# -------- SETTINGS --------
IMG_DIR = "/content/drive/MyDrive/GB-YOLOv7/yolov7/dataset-yolo_final/temp/images"
WEIGHTS = "/content/drive/MyDrive/GB-YOLOv7/yolov7/runs/train/gb_train_17-02-2026/weights/best.pt"  # or your runs/train/.../best.pt
YOLOV7_REPO = "/content/drive/MyDrive/GB-YOLOv7/yolov7"        # yolov7 repo root (must contain models/)
IMG_SIZE = 640
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
N_WARMUP = 30
N_RUN = 200
INCLUDE_PREPROCESS = True
# --------------------------

# Make YOLOv7 code importable
if YOLOV7_REPO not in sys.path:
    sys.path.append(YOLOV7_REPO)

from models.experimental import attempt_load  # YOLOv7 loader

# Speed/stability
torch.backends.cudnn.benchmark = True

def letterbox(img, size=640):
    """Simple square letterbox (keeps aspect ratio, pads to size x size)."""
    h, w = img.shape[:2]
    scale = min(size / h, size / w)
    nh, nw = int(round(h * scale)), int(round(w * scale))
    resized = cv2.resize(img, (nw, nh), interpolation=cv2.INTER_LINEAR)
    canvas = np.full((size, size, 3), 114, dtype=np.uint8)
    canvas[:nh, :nw] = resized
    return canvas

def preprocess(img):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    gray = cv2.medianBlur(gray, 5)
    clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))
    gray = clahe.apply(gray)
    return cv2.cvtColor(gray, cv2.COLOR_GRAY2BGR)

# Load images
files = sorted(glob.glob(os.path.join(IMG_DIR, "*")))
files = files[:min(N_RUN, len(files))]
if len(files) == 0:
    raise FileNotFoundError(f"No images found in: {IMG_DIR}")

# ---- Load model (PyTorch 2.6+ safe way for YOLOv7) ----
model = attempt_load(WEIGHTS, map_location=DEVICE)  # handles yolov7 checkpoints
model.eval().to(DEVICE)

# Warm-up
dummy = torch.zeros((1, 3, IMG_SIZE, IMG_SIZE), device=DEVICE)
with torch.no_grad():
    for _ in range(N_WARMUP):
        out = model(dummy)
        # touch output to ensure full compute path
        _ = out[0] if isinstance(out, (list, tuple)) else out
if DEVICE.startswith("cuda"):
    torch.cuda.synchronize()

times = []
with torch.no_grad():
    for fp in files:
        img = cv2.imread(fp)
        if img is None:
            continue

        t0 = time.perf_counter()

        if INCLUDE_PREPROCESS:
            img = preprocess(img)

        img = letterbox(img, IMG_SIZE)

        # BGR->RGB, HWC->CHW
        img = img[:, :, ::-1].transpose(2, 0, 1)
        img = np.ascontiguousarray(img)

        img = torch.from_numpy(img).to(DEVICE).float() / 255.0
        img = img.unsqueeze(0)  # (1,3,H,W)

        out = model(img)  # forward only
        _ = out[0] if isinstance(out, (list, tuple)) else out

        if DEVICE.startswith("cuda"):
            torch.cuda.synchronize()
        t1 = time.perf_counter()

        times.append((t1 - t0) * 1000)

times = np.array(times, dtype=np.float64)
print("Images measured:", len(times))
print("Mean latency (ms):", times.mean())
print("FPS:", 1000.0 / times.mean())
print("p50 (ms):", np.percentile(times, 50))
print("p90 (ms):", np.percentile(times, 90))
print("p99 (ms):", np.percentile(times, 99))


Fusing layers... 
IDetect.fuse
Images measured: 55
Mean latency (ms): 48.67610767272171
FPS: 20.543959815431258
p50 (ms): 47.675130000016
p90 (ms): 53.15024060000724
p99 (ms): 54.687356600002204


In [ ]:
import os, time, glob, sys
import cv2
import numpy as np
import torch

# -------- SETTINGS --------
IMG_DIR = "/content/drive/MyDrive/GB-YOLOv7/yolov7/dataset-yolo_final/temp/images"
WEIGHTS = "/content/drive/MyDrive/GB-YOLOv7/yolov7/runs/train/gb_train_17-02-2026/weights/best.pt"  # or your runs/train/.../best.pt
YOLOV7_REPO = "/content/drive/MyDrive/GB-YOLOv7/yolov7"        # yolov7 repo root (must contain models/)
IMG_SIZE = 640
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
N_WARMUP = 30
N_RUN = 200
INCLUDE_PREPROCESS = False
# --------------------------

# Make YOLOv7 code importable
if YOLOV7_REPO not in sys.path:
    sys.path.append(YOLOV7_REPO)

from models.experimental import attempt_load  # YOLOv7 loader

# Speed/stability
torch.backends.cudnn.benchmark = True

def letterbox(img, size=640):
    """Simple square letterbox (keeps aspect ratio, pads to size x size)."""
    h, w = img.shape[:2]
    scale = min(size / h, size / w)
    nh, nw = int(round(h * scale)), int(round(w * scale))
    resized = cv2.resize(img, (nw, nh), interpolation=cv2.INTER_LINEAR)
    canvas = np.full((size, size, 3), 114, dtype=np.uint8)
    canvas[:nh, :nw] = resized
    return canvas

def preprocess(img):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    gray = cv2.medianBlur(gray, 5)
    clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))
    gray = clahe.apply(gray)
    return cv2.cvtColor(gray, cv2.COLOR_GRAY2BGR)

# Load images
files = sorted(glob.glob(os.path.join(IMG_DIR, "*")))
files = files[:min(N_RUN, len(files))]
if len(files) == 0:
    raise FileNotFoundError(f"No images found in: {IMG_DIR}")

# ---- Load model (PyTorch 2.6+ safe way for YOLOv7) ----
model = attempt_load(WEIGHTS, map_location=DEVICE)  # handles yolov7 checkpoints
model.eval().to(DEVICE)

# Warm-up
dummy = torch.zeros((1, 3, IMG_SIZE, IMG_SIZE), device=DEVICE)
with torch.no_grad():
    for _ in range(N_WARMUP):
        out = model(dummy)
        # touch output to ensure full compute path
        _ = out[0] if isinstance(out, (list, tuple)) else out
if DEVICE.startswith("cuda"):
    torch.cuda.synchronize()

times = []
with torch.no_grad():
    for fp in files:
        img = cv2.imread(fp)
        if img is None:
            continue

        t0 = time.perf_counter()

        if INCLUDE_PREPROCESS:
            img = preprocess(img)

        img = letterbox(img, IMG_SIZE)

        # BGR->RGB, HWC->CHW
        img = img[:, :, ::-1].transpose(2, 0, 1)
        img = np.ascontiguousarray(img)

        img = torch.from_numpy(img).to(DEVICE).float() / 255.0
        img = img.unsqueeze(0)  # (1,3,H,W)

        out = model(img)  # forward only
        _ = out[0] if isinstance(out, (list, tuple)) else out

        if DEVICE.startswith("cuda"):
            torch.cuda.synchronize()
        t1 = time.perf_counter()

        times.append((t1 - t0) * 1000)

times = np.array(times, dtype=np.float64)
print("Images measured:", len(times))
print("Mean latency (ms):", times.mean())
print("FPS:", 1000.0 / times.mean())
print("p50 (ms):", np.percentile(times, 50))
print("p90 (ms):", np.percentile(times, 90))
print("p99 (ms):", np.percentile(times, 99))


Fusing layers... 
IDetect.fuse
Images measured: 55
Mean latency (ms): 39.59379461819233
FPS: 25.256482982828977
p50 (ms): 39.57804200001647
p90 (ms): 40.94972000000325
p99 (ms): 41.45063067991259


In [ ]:
import os
import cv2
import numpy as np
from tqdm import tqdm

# ==============================
# CONFIGURATION
# ==============================
INPUT_DIR = "/content/drive/MyDrive/GB-YOLOv7/yolov7/dataset-yolo_final/malignant_only/images"   # <-- change this
OUTPUT_DIR = "/content/drive/MyDrive/GB-YOLOv7/yolov7/preprocess-test-images_v2" # <-- change this

MEDIAN_KERNEL = 3          # median filter kernel size
CLAHE_CLIP = 1.5
CLAHE_TILE = (8, 8)

# ==============================
# CREATE OUTPUT DIR
# ==============================
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ==============================
# INITIALIZE CLAHE
# ==============================
clahe = cv2.createCLAHE(
    clipLimit=CLAHE_CLIP,
    tileGridSize=CLAHE_TILE
)

# ==============================
# PROCESS IMAGES
# ==============================
image_files = [f for f in os.listdir(INPUT_DIR)
               if f.lower().endswith(('.png', '.jpg', '.jpeg'))]

print(f"Found {len(image_files)} images")

for fname in tqdm(image_files):
    img_path = os.path.join(INPUT_DIR, fname)
    img = cv2.imread(img_path)

    if img is None:
        continue

    # ---- Median Filtering ----
    img_median = cv2.medianBlur(img, MEDIAN_KERNEL)

    # ---- CLAHE in LAB space ----
    lab = cv2.cvtColor(img_median, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)

    l_clahe = clahe.apply(l)

    lab_clahe = cv2.merge((l_clahe, a, b))
    img_clahe = cv2.cvtColor(lab_clahe, cv2.COLOR_LAB2BGR)

    # ---- Save Output ----
    save_path = os.path.join(OUTPUT_DIR, fname)
    cv2.imwrite(save_path, img_clahe)

print("Preprocessing complete.")
print(f"Processed images saved to: {OUTPUT_DIR}")


Found 37 images


100%|██████████| 37/37 [00:02<00:00, 13.84it/s]

Preprocessing complete.
Processed images saved to: /content/drive/MyDrive/GB-YOLOv7/yolov7/preprocess-test-images_v2


In [ ]:
!pip install grad-cam


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 77.2 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for grad-cam: filename=grad_cam-1.5.5-py3-none-any.whl size=44285 sha256=3fcc1d426d6f02818c9fd40e1923e685c5d22b0e454541cc014da0bedb8d67b6
  Stored in directory: /root/.cache/pip/wheels/fb/3b/09/2afc520f3d69bc26ae6bd87416759c820a3f7d05c1a077bbf6
Successfully built grad-cam


In [ ]:
import torch

model = torch.load(
    "/content/drive/MyDrive/GB-YOLOv7/yolov7/runs/train/gb_train_17-02-2026/weights/best.pt",
    map_location="cpu",
    weights_only=False
)["model"]

# print named modules
for i, (name, module) in enumerate(model.model.named_modules()):
    print(i, name)

0 
1 0
2 0.conv
3 0.bn
4 0.act
5 1
6 1.conv
7 1.bn
8 1.act
9 2
10 2.conv
11 2.bn
12 2.act
13 3
14 3.conv
15 3.bn
16 3.act
17 4
18 4.conv
19 4.bn
20 4.act
21 5
22 5.conv
23 5.bn
24 5.act
25 6
26 6.conv
27 6.bn
28 6.act
29 7
30 7.conv
31 7.bn
32 7.act
33 8
34 8.conv
35 8.bn
36 8.act
37 9
38 9.conv
39 9.bn
40 9.act
41 10
42 11
43 11.conv
44 11.bn
45 11.act
46 12
47 12.channel_attention
48 12.channel_attention.0
49 12.channel_attention.1
50 12.channel_attention.2
51 12.spatial_attention
52 12.spatial_attention.0
53 12.spatial_attention.1
54 12.spatial_attention.2
55 12.spatial_attention.3
56 12.spatial_attention.4
57 13
58 13.m
59 14
60 14.conv
61 14.bn
62 14.act
63 15
64 15.conv
65 15.bn
66 15.act
67 16
68 16.conv
69 16.bn
70 16.act
71 17
72 18
73 18.conv
74 18.bn
75 18.act
76 19
77 19.conv
78 19.bn
79 19.act
80 20
81 20.conv
82 20.bn
83 20.act
84 21
85 21.conv
86 21.bn
87 21.act
88 22
89 22.conv
90 22.bn
91 22.act
92 23
93 23.conv
94 23.bn
95 23.act
96 24
97 25
98 25.conv
99 25.bn
100 25

In [ ]:
!python main_gradcam.py \
  --model-path "/content/drive/MyDrive/GB-YOLOv7/yolov7/runs/train/gb_train_17-02-2026/weights/best.pt" \
  --img-path "/content/drive/MyDrive/GB-YOLOv7/yolov7/preprocess-test-images_v1/im00067.jpg" \
  --output-dir "/content/drive/MyDrive/GB-YOLOv7/yolov7/gradcam_out/" \
  --method gradcam \
  --device cuda